In [ ]:
"""
GOOGLE COLAB: GP Learning Animation - 1D b vs Str/w
====================================================
Watch how the Gaussian Process learns the effect of web thickness (b)
on strength-to-weight ratio as beams are tested one by one.

4D input space: (b, r, dH, B) to properly handle varying flange widths.
Warm-started kernel to prevent degenerate flat solutions.
Heteroscedastic noise: higher near the lateral-torsional buckling boundary.

INSTRUCTIONS:
1. Run all cells (Runtime > Run all)
2. Animation will be saved and displayed at the end
3. Adjust parameters in CONFIGURATION section below
"""

# ==========================================
# CONFIGURATION - ADJUST THESE!
# ==========================================

INITIAL_DATA_SIZE = 1
STEP_SIZE = 1
PLOT_EVERY = 1

LENGTH_SCALE_MODE = 'free'

ANIMATION_FPS = 3
ANIMATION_FILENAME = 'gp_learning_b_vs_strw.gif'

FIGURE_SIZE = (10, 6)
DPI = 100

# ==========================================
# INSTALLATION & IMPORTS
# ==========================================

!pip install -q scikit-learn matplotlib pandas numpy Pillow imageio

import pandas as pd
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel
import matplotlib.pyplot as plt
from IPython.display import Image as IPImage
import warnings
warnings.filterwarnings("ignore")

print("✓ Packages installed and imported")

# ==========================================
# PHYSICS CONSTANTS & 4D BOUNDS
# ==========================================

TOTAL_HEIGHT = 25.0
B_FIXED = 16.0
MIN_WEB_THICKNESS = 0.8
MIN_FLANGE_WIDTH = 8.0
MAX_WEB_RATIO = 2/3
MATERIAL_DENSITY = 1240
LENGTH_M = 0.2023
YIELD_STRENGTH = 76000000
E_MODULUS = 2.5e9
G_MODULUS = E_MODULUS / 2.6
C1_3PT = 1.35

BOUNDS_4D = {
    'b': (0.8, 10.67),
    'r': (0.0, 4.0),
    'delta_H': (-6.0, 4.0),
    'B': (8.0, 16.0),
}
PARAM_KEYS = list(BOUNDS_4D.keys())

print("✓ Physics constants loaded (4D space: b, r, dH, B)")

# ==========================================
# PHYSICS CALCULATIONS
# ==========================================

def calc_I(H, h, B, b):
    """Strong-axis moment of inertia (m^4)"""
    H_m, h_m, B_m, b_m = H/1000, h/1000, B/1000, b/1000
    return (H_m**3 * b_m)/12 + 2*((h_m**3 * B_m)/12 + h_m*B_m*((H_m+h_m)/2)**2)

def calc_Iy(H, h, B, b):
    """Weak-axis moment of inertia (m^4)"""
    H_m, h_m, B_m, b_m = H/1000, h/1000, B/1000, b/1000
    return (H_m * b_m**3)/12 + 2*(h_m * B_m**3)/12

def calc_J(H, h, B, b):
    """Torsional constant, thin-wall approximation (m^4)"""
    H_m, h_m, B_m, b_m = H/1000, h/1000, B/1000, b/1000
    return (H_m * b_m**3 + 2*B_m * h_m**3) / 3

def calc_mass(H, h, B, b):
    H_m, h_m, B_m, b_m = H/1000, h/1000, B/1000, b/1000
    return MATERIAL_DENSITY * LENGTH_M * (H_m*b_m + 2*h_m*B_m) * 1000

def calc_strength(H, h, B, b):
    return (4 * YIELD_STRENGTH * calc_I(H, h, B, b)) / (0.0125 * LENGTH_M)

def calc_str_w(H, B, b):
    h = (TOTAL_HEIGHT - H) / 2.0
    return calc_strength(H, h, B, b) / calc_mass(H, h, B, b)

def calc_Mcr(H, h, B, b):
    """Critical lateral-torsional buckling moment for 3-point bending"""
    Iy = calc_Iy(H, h, B, b)
    J = calc_J(H, h, B, b)
    if Iy <= 0 or J <= 0:
        return 0.0
    return (C1_3PT * np.pi / LENGTH_M) * np.sqrt(E_MODULUS * Iy * G_MODULUS * J)

def calc_Myield(H, h, B, b):
    """Yield moment = sigma_y * I_x / y_max"""
    Ix = calc_I(H, h, B, b)
    y_max = (H/1000 + h/1000)  # distance from neutral axis to extreme fiber (full half-depth)
    if y_max <= 0:
        return 0.0
    return YIELD_STRENGTH * Ix / y_max

def stability_ratio(H, h, B, b):
    """R = M_cr / M_yield. R~1 is the tipping boundary."""
    Mcr = calc_Mcr(H, h, B, b)
    My = calc_Myield(H, h, B, b)
    if My <= 0:
        return 0.0
    return Mcr / My

def stability_ratio_from_4d(x_4d):
    """Compute stability ratio from (b, r, dH, B) parameterization"""
    b, r, dH, B = x_4d
    H_phys = find_H_opt(b, B)
    H = H_phys + dH
    H = np.clip(H, 12.0, 23.4)
    h = (TOTAL_HEIGHT - H) / 2.0
    if h <= 0 or h > 6.5:
        return 0.0
    return stability_ratio(H, h, B, b)

def find_H_opt(b, B=B_FIXED):
    from scipy.optimize import minimize_scalar
    def obj(H):
        if H < 12.0 or H > 23.4: return 1e10
        h = (TOTAL_HEIGHT - H) / 2.0
        if h < 0 or h > 6.5: return 1e10
        return -calc_str_w(H, B, b)
    return minimize_scalar(obj, bounds=(12.0, 23.4), method='bounded').x

print("✓ Lateral stability functions defined")

# ==========================================
# HETEROSCEDASTIC NOISE MODEL
# ==========================================

def get_noise_variance(R):
    """
    Noise variance in log(Str/w) space as a function of stability ratio R.
    Peak noise at R~1 (tipping boundary), low noise on both sides.
    Calibrated from LHS6 original vs repeat: delta_log ~ 0.083, sigma ~ 0.06.
    """
    sigma_base = 0.001
    sigma_peak = 0.004
    width = 0.5
    log_R = np.log(np.maximum(R, 1e-6))
    bump = np.exp(-0.5 * (log_R / width)**2)
    return sigma_base + sigma_peak * bump

def get_alpha_array(X_4d):
    """Compute per-point noise variance from stability ratio"""
    alphas = np.array([get_noise_variance(stability_ratio_from_4d(x)) for x in X_4d])
    return alphas

print("✓ Heteroscedastic noise model defined")

# ==========================================
# TRANSFORMS
# ==========================================

def transform_to_4d(X_raw):
    """Transform raw (H, B, b, r) to (b, r, dH, B)"""
    X_4d = []
    for i in range(len(X_raw)):
        H, B, b, r = X_raw[i]
        H_phys = find_H_opt(b, B)
        delta_H = H - H_phys
        X_4d.append([b, r, delta_H, B])
    return np.array(X_4d)

def normalize(X):
    X_n = np.copy(X).astype(float)
    for i, k in enumerate(PARAM_KEYS):
        X_n[:, i] = (X[:, i] - BOUNDS_4D[k][0]) / (BOUNDS_4D[k][1] - BOUNDS_4D[k][0])
    return X_n

def denormalize(X_n):
    X = np.copy(X_n)
    for i, k in enumerate(PARAM_KEYS):
        X[:, i] = X_n[:, i] * (BOUNDS_4D[k][1] - BOUNDS_4D[k][0]) + BOUNDS_4D[k][0]
    return X

print("✓ Transform functions defined")

# ==========================================
# GP TRAINING WITH WARM START + HETEROSCEDASTIC NOISE
# ==========================================

prev_kernel = None

def train_gp(X_4d, y):
    """Train GP with warm-start kernel and per-point heteroscedastic noise"""
    global prev_kernel
    X_norm = normalize(X_4d)
    y_log = np.log(y)
    y_mean, y_cent = np.mean(y_log), y_log - np.mean(y_log)

    alpha_array = get_alpha_array(X_4d)

    if prev_kernel is not None:
        kernel = prev_kernel
    else:
        kernel = ConstantKernel(1.0, (0.1, 10.0)) * Matern(
            length_scale=[0.5]*4, length_scale_bounds=(0.1, 2.0), nu=2.5)

    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10,
                                   alpha=alpha_array, normalize_y=False)
    gp.fit(X_norm, y_cent)

    prev_kernel = gp.kernel_

    return gp, X_norm, y_cent, y_mean

print("✓ GP training: warm-start + heteroscedastic alpha")

# ==========================================
# DATA LOADING
# ==========================================

def load_data():
    import urllib.request, io
    DATA_URL = "https://raw.githubusercontent.com/andrewvoss8-boop/core-me-data-science-activities-public/main/data/I_beam_data.csv"
    print(f"Downloading data from GitHub...")
    with urllib.request.urlopen(DATA_URL) as response:
        csv_data = response.read().decode('utf-8')
    df = pd.read_csv(io.StringIO(csv_data))
    param_names = ['H_web_height', 'B_flange_width', 'b_web_thick', 'r_fillet']
    df_clean = df[param_names + ['Str/w (N/g)']].dropna()

    def meets_constraints(row):
        H, B, b, r = row['H_web_height'], row['B_flange_width'], row['b_web_thick'], row['r_fillet']
        h = (TOTAL_HEIGHT - H) / 2.0
        if B < MIN_FLANGE_WIDTH or b < MIN_WEB_THICKNESS or b > MAX_WEB_RATIO * B: return False
        if r < 0 or r > (B - b) / 2.0: return False
        if h < 0 or h > 6.5: return False
        return True

    df_ok = df_clean[df_clean.apply(meets_constraints, axis=1)].copy()
    print(f"✓ Loaded {len(df_ok)} valid beams, Str/w: [{df_ok['Str/w (N/g)'].min():.2f}, {df_ok['Str/w (N/g)'].max():.2f}]")
    return df_ok

df = load_data()
X_raw_all = df[['H_web_height', 'B_flange_width', 'b_web_thick', 'r_fillet']].values
y_all = df['Str/w (N/g)'].values
X_4d_all = transform_to_4d(X_raw_all)
X_norm_all = normalize(X_4d_all)

final_best_idx = np.argmax(y_all)
final_best_beam = X_4d_all[final_best_idx]
final_best_str_w = y_all[final_best_idx]
slice_point_norm = normalize(final_best_beam.reshape(1, -1))[0]

# Print stability ratios for all beams
print(f"\n✓ Data prepared ({len(y_all)} beams)")
print(f"  Best beam: b={final_best_beam[0]:.2f}, r={final_best_beam[1]:.2f}, "
      f"dH={final_best_beam[2]:.2f}, B={final_best_beam[3]:.2f} -> Str/w={final_best_str_w:.2f}")
print(f"\n  Stability ratios and noise for each beam:")
print(f"  {'#':>3} {'b':>5} {'r':>5} {'dH':>6} {'B':>5} {'Str/w':>6} {'R':>6} {'noise':>8}")
for i in range(len(y_all)):
    R = stability_ratio_from_4d(X_4d_all[i])
    nv = get_noise_variance(R)
    print(f"  {i+1:3d} {X_4d_all[i,0]:5.2f} {X_4d_all[i,1]:5.2f} {X_4d_all[i,2]:6.2f} "
          f"{X_4d_all[i,3]:5.1f} {y_all[i]:6.2f} {R:6.3f} {nv:8.5f}")

# ==========================================
# FRAME GENERATION
# ==========================================

def get_point_sizes(X_norm, slice_point_norm):
    distances = np.linalg.norm(X_norm - slice_point_norm, axis=1)
    sizes = np.zeros_like(distances)
    colors = np.zeros_like(distances)
    for mask, sz, cl in [
        (distances < 0.15, 250, 0.9),
        ((distances >= 0.15) & (distances < 0.30), 180, 0.7),
        ((distances >= 0.30) & (distances < 0.50), 120, 0.5),
        ((distances >= 0.50) & (distances < 0.75), 70, 0.3),
        (distances >= 0.75, 30, 0.1),
    ]:
        sizes[mask] = sz
        colors[mask] = cl
    return sizes, distances, colors

def create_frame(ax, n_beams, X_4d_all, y_all, X_norm_all, slice_point_norm):
    X_4d_current = X_4d_all[:n_beams]
    y_current = y_all[:n_beams]

    gp, X_norm_current, y_cent, y_mean = train_gp(X_4d_current, y_current)

    scales = gp.kernel_.k2.length_scale
    scale_str = f"b={scales[0]:.2f}, r={scales[1]:.2f}, dH={scales[2]:.2f}, B={scales[3]:.2f}"

    current_best_idx = np.argmax(y_current)
    current_best_str_w = y_current[current_best_idx]

    ax.clear()

    # Predict along b, holding r/dH/B at slice point
    X_test_norm = np.tile(slice_point_norm, (100, 1))
    X_test_norm[:, 0] = np.linspace(0, 1, 100)
    X_test_real = denormalize(X_test_norm)

    mu, sig = gp.predict(X_test_norm, return_std=True)
    mu_r = np.exp(mu + y_mean)
    epistemic_std = np.exp(mu + y_mean) * sig

    # Position-dependent aleatory noise along the slice
    test_noise_var = np.array([get_noise_variance(stability_ratio_from_4d(x)) for x in X_test_real])
    aleatory_std = np.exp(mu + y_mean) * np.sqrt(test_noise_var)
    total_std = np.sqrt(epistemic_std**2 + aleatory_std**2)

    b_vals = X_test_real[:, 0]

    ax.plot(b_vals, mu_r, 'b-', lw=3, label='GP mean', zorder=5)
    ax.fill_between(b_vals, mu_r - 2*total_std, mu_r + 2*total_std,
                    alpha=0.25, color='orange', label='±2σ aleatory', zorder=3)
    ax.fill_between(b_vals, mu_r - 2*epistemic_std, mu_r + 2*epistemic_std,
                    alpha=0.35, color='blue', label='±2σ epistemic', zorder=4)

    # Future data
    future_mask = np.arange(len(y_all)) >= n_beams
    X_den_all = denormalize(X_norm_all)
    if np.any(future_mask):
        ax.scatter(X_den_all[future_mask, 0], y_all[future_mask],
                  c='lightgray', s=30, alpha=0.3, ec='gray', lw=0.5,
                  zorder=2, label='Future data')

    # Current data sized by 4D distance from slice
    point_sizes, _, point_colors = get_point_sizes(X_norm_current, slice_point_norm)
    X_den_current = denormalize(X_norm_current)
    ax.scatter(X_den_current[:, 0], y_current, s=point_sizes,
              c=point_colors, cmap='Reds', vmin=0, vmax=1, alpha=0.8,
              ec='black', lw=0.5, zorder=6, label=f'Training data (n={n_beams})')

    ax.scatter([X_den_current[current_best_idx, 0]], [current_best_str_w],
              s=600, marker='*', c='gold', ec='black', lw=2.5, zorder=10,
              label='Current best')

    ax.scatter([final_best_beam[0]], [final_best_str_w],
              s=500, marker='D', c='lime', ec='black', lw=2, zorder=9,
              label='Final best (slice)')

    ax.set_xlabel('b_web (mm)', fontsize=14, weight='bold')
    ax.set_ylabel('Str/w (N/g)', fontsize=14, weight='bold')
    ax.set_title(f'GP Learning (4D, heteroscedastic): b vs Str/w - Beam {n_beams}/{len(y_all)}\n'
                f'Best: {current_best_str_w:.2f} N/g | LS: {scale_str}',
                fontsize=11, weight='bold', pad=15)
    ax.legend(fontsize=9, loc='lower right', framealpha=0.9)
    ax.grid(alpha=0.3, linestyle='--')
    ax.set_xlim(BOUNDS_4D['b'][0], BOUNDS_4D['b'][1])

    y_min, y_max = y_all.min(), y_all.max()
    y_range = y_max - y_min
    ax.set_ylim(y_min - 0.1*y_range, y_max + 0.1*y_range)
    return ax

print("✓ Frame generation defined (4D + warm-start + heteroscedastic)")

# ==========================================
# CREATE ANIMATION
# ==========================================

print("\n" + "="*70)
print("GENERATING ANIMATION")
print("="*70)

prev_kernel = None

frame_indices = list(range(INITIAL_DATA_SIZE, len(y_all) + 1, STEP_SIZE))
if frame_indices[-1] != len(y_all):
    frame_indices.append(len(y_all))
frame_indices = [idx for i, idx in enumerate(frame_indices) if (i % PLOT_EVERY) == 0]

print(f"  Frames: {len(frame_indices)}, FPS: {ANIMATION_FPS}")
print(f"  Slice: b={final_best_beam[0]:.2f}, r={final_best_beam[1]:.2f}, "
      f"dH={final_best_beam[2]:.2f}, B={final_best_beam[3]:.2f}")
print(f"\nGenerating frames...")

fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)
plt.tight_layout()

frames = []
for i, n_beams in enumerate(frame_indices):
    print(f"  Frame {i+1}/{len(frame_indices)}: {n_beams} beams", end='\r')
    create_frame(ax, n_beams, X_4d_all, y_all, X_norm_all, slice_point_norm)
    import io
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=DPI, bbox_inches='tight')
    buf.seek(0)
    from PIL import Image
    frames.append(Image.open(buf).copy())
    buf.close()

plt.close(fig)
print(f"\n✓ Generated {len(frames)} frames")

print(f"Saving: {ANIMATION_FILENAME}")
import imageio
imageio.mimsave(ANIMATION_FILENAME, [np.array(f) for f in frames], fps=ANIMATION_FPS, loop=0)
print(f"✓ Saved!")

display(IPImage(ANIMATION_FILENAME))
print("✓ Done!")


In [ ]:
# ==========================================
# SENSITIVITY ANALYSIS: Which knob controls tipping?
# ==========================================
# Run this AFTER the animation cell to use the loaded data and functions.

print("="*70)
print("SENSITIVITY ANALYSIS: Stability Ratio vs Design Parameters")
print("="*70)

# Evaluate at the best beam design
x_best = final_best_beam.copy()  # (b, r, dH, B)
R_best = stability_ratio_from_4d(x_best)
print(f"\nBest beam: b={x_best[0]:.2f}, r={x_best[1]:.2f}, dH={x_best[2]:.2f}, B={x_best[3]:.1f}")
print(f"Stability ratio R = {R_best:.4f}")
print(f"Noise variance at this R: {get_noise_variance(R_best):.6f}")

# Finite difference sensitivities
eps = 1e-4
param_names = ['b', 'r', 'dH', 'B']
param_ranges = [BOUNDS_4D[k][1] - BOUNDS_4D[k][0] for k in PARAM_KEYS]

print(f"\n{'Param':>6} {'dR/dx':>10} {'dR/dx_norm':>12} {'range':>7}  interpretation")
print("-"*70)

sensitivities = {}
for dim in range(4):
    x_plus = x_best.copy()
    x_minus = x_best.copy()
    x_plus[dim] += eps
    x_minus[dim] -= eps
    R_plus = stability_ratio_from_4d(x_plus)
    R_minus = stability_ratio_from_4d(x_minus)
    dR_dx = (R_plus - R_minus) / (2 * eps)
    dR_dx_norm = dR_dx * param_ranges[dim]
    sensitivities[param_names[dim]] = {'raw': dR_dx, 'normalized': dR_dx_norm}

    if abs(dR_dx_norm) < 0.01:
        interp = "negligible"
    elif dR_dx_norm > 0:
        interp = f"increasing {param_names[dim]} STABILIZES"
    else:
        interp = f"increasing {param_names[dim]} DESTABILIZES"

    print(f"{param_names[dim]:>6} {dR_dx:10.4f} {dR_dx_norm:12.4f} {param_ranges[dim]:7.2f}  {interp}")

# Rank by magnitude
ranked = sorted(sensitivities.items(), key=lambda x: abs(x[1]['normalized']), reverse=True)
print(f"\nRanked by effectiveness (normalized |dR/dx| over full parameter range):")
for i, (name, vals) in enumerate(ranked):
    print(f"  {i+1}. {name}: |dR_norm| = {abs(vals['normalized']):.4f}")

# Sweep each parameter and plot R
print(f"\nGenerating stability ratio sweep plots...")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Stability Ratio R vs Each Parameter\n'
             '(other params fixed at best beam, dashed line = R=1 tipping boundary)',
             fontsize=14, weight='bold')

for dim, ax in enumerate(axes.flatten()):
    n_pts = 200
    x_sweep = np.tile(x_best, (n_pts, 1))
    lo, hi = BOUNDS_4D[PARAM_KEYS[dim]]
    sweep_vals = np.linspace(lo, hi, n_pts)
    x_sweep[:, dim] = sweep_vals

    R_vals = np.array([stability_ratio_from_4d(x) for x in x_sweep])
    noise_vals = np.array([get_noise_variance(R) for R in R_vals])

    ax.plot(sweep_vals, R_vals, 'b-', lw=2, label='R (stability ratio)')
    ax.axhline(1.0, color='red', ls='--', lw=1.5, alpha=0.7, label='R=1 (tipping boundary)')
    ax.axvline(x_best[dim], color='green', ls='--', lw=1.5, alpha=0.7, label='Best beam')

    ax2 = ax.twinx()
    ax2.plot(sweep_vals, noise_vals * 1000, 'orange', lw=1.5, alpha=0.7, label='noise var (x1000)')
    ax2.set_ylabel('noise var (x1000)', fontsize=10, color='orange')
    ax2.tick_params(axis='y', labelcolor='orange')

    ax.set_xlabel(f'{param_names[dim]} (mm)', fontsize=12)
    ax.set_ylabel('Stability Ratio R', fontsize=12)
    ax.set_title(f'{param_names[dim]}: dR_norm = {sensitivities[param_names[dim]]["normalized"]:.3f}',
                 fontsize=12, weight='bold')
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('stability_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

# Cross-reference with GP length scales
print(f"\nGP learned length scales (from final frame):")
scales = prev_kernel.k2.length_scale
for dim in range(4):
    print(f"  {param_names[dim]:>3}: ls = {scales[dim]:.3f}  "
          f"(smaller = GP sees more variation along this axis)")

print(f"\n{'='*70}")
print("INTERPRETATION")
print("="*70)
print("""
R > 1: beam yields before tipping (laterally stable)
R < 1: beam tips before yielding (laterally unstable)
R ~ 1: boundary regime, failure mode is stochastic (highest noise)

The orange noise curve peaks where R crosses 1, showing where
repeated tests of the same design would give the most scatter.

dR_norm tells you: if you sweep a parameter across its full range
(holding others fixed), how much does R change? Larger magnitude
means that parameter is a more effective lever for controlling
whether the beam tips or yields.
""")
